1. LeNet(LeNet-5)由两个部分组成：卷积编码器和全连接层密集块

In [3]:
import torch
from torch import nn
from d2l import torch as d2l

class Reshape(torch.nn.Module):
    def forward(self, x):
        return x.view(-1, 1, 28, 28)
    
net = torch.nn.Sequential(
    # (batch_size, 1, 28, 28) - (1, 28, 28) 单通道28x28
    Reshape(), 
    # kernel_size 需要自己算
    # 28+4-5/1+1=28 （6,28,28） - (6,28,28) 6通道
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    # 均值池化层 (6,28,28) - (6,14,14)
    nn.AvgPool2d(2, stride=2),
    # (6,14,14) - (16,10,10) 14-5/1+1=10 16通道
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    # (16,10,10) - (16,5,5)
    nn.AvgPool2d(2, stride=2),
    # 4x1 -> 1维 16x5x5
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10)  
)

2. 检查模型

In [4]:
X = torch.rand(size=(1, 1, 28, 28), dtype=torch.float32)
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t',X.shape)

Reshape output shape:	 torch.Size([1, 1, 28, 28])
Conv2d output shape:	 torch.Size([1, 6, 28, 28])
Sigmoid output shape:	 torch.Size([1, 6, 28, 28])
AvgPool2d output shape:	 torch.Size([1, 6, 14, 14])
Conv2d output shape:	 torch.Size([1, 16, 10, 10])
Sigmoid output shape:	 torch.Size([1, 16, 10, 10])
AvgPool2d output shape:	 torch.Size([1, 16, 5, 5])
Flatten output shape:	 torch.Size([1, 400])
Linear output shape:	 torch.Size([1, 120])
Sigmoid output shape:	 torch.Size([1, 120])
Linear output shape:	 torch.Size([1, 84])
Sigmoid output shape:	 torch.Size([1, 84])
Linear output shape:	 torch.Size([1, 10])


3.LeNet在 Fashion-MNIST 数据集中的表现

In [5]:
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size=batch_size)

4.对 evaluate_accuracy函数进行轻微修改

In [1]:
def evaluate_accuracy_gpu(net, data_iter, device=None):
    """使用gpu计算"""
    if isinstance(net, torch.nn.Module):
        net.eval()
        if not device:
            device = next(iter(net.parameters())).device
    metric = d2l.Accumulator(2)
    for X, y in data_iter:
        if isinstance(X, list):
            X = [x.to(device) for x in X]
        else:
            X = X.to(device)
        y = y.to(device)
        metric.add(d2l.accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]